## Notebook23c

In this notebook, we will see how to call an API to use a large language model that cannot easily run locally on our machine.

### Setup

Run all of the following before starting the notebook.

In [ ]:
! wget -q -nc https://raw.githubusercontent.com/taylor-arnold/fds-py-nb/refs/heads/main/funs.py

In [ ]:
import json
import os
import time
from pathlib import Path

import numpy as np
import polars as pl
from openai import OpenAI
from pydantic import BaseModel, Field
from enum import Enum

from funs import *
from plotnine import *
from polars import col as c

theme_set(theme_minimal())
pl.Config(tbl_rows=25)

ub = "https://raw.githubusercontent.com/taylor-arnold/fds-py-nb/refs/heads/main/"

In [ ]:
agnews = pl.read_parquet(ub + "data/agnews_pca.parquet")

### Overview

In this notebook we use a large language model as a **zero-shot text classifier**. Rather than training a model from scratch (as we did with our baby LLM), we send text to a pre-trained model through an API and ask it to assign a category — no training data, no fine-tuning, no gradient descent.

The interesting parts are:

1. **Structured outputs with Pydantic** — instead of hoping the model returns nicely formatted text, we define a schema that forces the response into a predictable shape we can work with programmatically.
2. **Evaluation** — we compare the model's predictions against the true labels in AG News using standard classification metrics (precision, recall, F1).
3. **Caching** — API calls cost money and take time, so we save results to disk and reload them on subsequent runs.

The AG News dataset has four categories: **World**, **Sports**, **Business**, and **Sci/Tech**. Each article was originally labeled by humans, so we have ground truth to compare against.

### Connecting to the OpenAI API

The `openai` Python package handles all communication with OpenAI's models. You create a client object, and then call methods on it to send requests. The API key is already loaded in our environment.

That's it — the client picks up the `OPENAI_API_KEY` environment variable automatically. No key management code needed in the notebook.

### The AG News Dataset

We have the same AG News dataset from our baby LLM notebook. Let's take a quick look at its structure.

### Sampling a Subset

Classifying the full dataset through an API would be slow and expensive. Instead we'll pull a small stratified subset (~100 rows) so each category is represented equally.

### Defining Structured Output with Pydantic

Here's the core idea: instead of asking the model to return free text and then parsing it ourselves, we define a **Pydantic model** that describes exactly what we want back. The OpenAI API can enforce this schema, guaranteeing that the response is valid structured data.

First, we define an enum for the four categories. This constrains the model to only these choices — it literally cannot return anything else.

Now we define a Pydantic model for the full response. Beyond just the category, we ask for a brief `reasoning` field. This serves two purposes: it gives us insight into *why* the model chose a category, and it actually tends to improve accuracy because the model "thinks through" its answer before committing.

Let's look at the JSON schema that Pydantic generates from this. This is exactly what gets sent to the API to enforce the response structure.

### Making a Single Classification

Before we classify hundreds of articles, let's walk through a single API call to see how all the pieces fit together.

We call `client.beta.chat.completions.parse()` instead of the usual `client.chat.completions.create()`. The `parse` method accepts a `response_format` parameter that takes our Pydantic model and enforces the schema on the response.

The response is a proper `ArticleClassification` object — not a string we need to parse. The `category` field is guaranteed to be one of our four enum values. This is what makes structured outputs so powerful for data pipelines: no regex, no "please format your answer as JSON", no error handling for malformed responses.

### Classifying a Batch with Caching

Now we scale this up to our full samples. The function below classifies a list of articles and caches the results to a JSON file. On subsequent runs it loads from the cache instead of hitting the API again.

A few things to note about this function:

- **Caching**: results are saved as a JSON file. If the file exists on the next run, we skip the API entirely. Delete the cache file to re-classify.
- **Rate limiting**: the `time.sleep(delay)` adds a small pause between requests to avoid hitting rate limits.
- **Error handling**: if a single request fails, we record the error and continue rather than crashing the whole batch.

### Running the Small Sample

Let's classify the small sample first. This should take about 30 seconds.

We add the predictions back to our Polars DataFrame as a new column.

### Evaluating Performance

How well did the model do? Let's start with simple accuracy, then look at per-class precision, recall, and F1.

For a model that has never seen a single training example from this dataset, these numbers are typically very strong — often above 85% accuracy. The model is relying entirely on its pre-training knowledge of what "business news" or "sports news" looks like.

### Looking at Mistakes

The misclassifications are often more interesting than the correct ones. Let's see where the model struggled.

You'll often find that the mistakes are on genuinely ambiguous articles — a story about a tech company's stock price could reasonably be "Business" or "Sci/Tech", for example. This is a useful discussion point: the model isn't always *wrong* when it disagrees with the label; sometimes the original label is debatable.

### Cost and Latency

A practical consideration: how much did this cost? The `gpt-4o-mini` model is very cheap, but it's still worth tracking.

### What We Learned

This notebook demonstrated a fundamentally different approach to NLP compared to training your own model:

**Zero-shot classification** — we never showed the model a single labeled example from AG News. It classified articles based entirely on its pre-existing understanding of language and news categories. This is only possible because the model was pre-trained on a massive corpus that included similar content.

**Structured outputs** — by defining a Pydantic schema, we got guaranteed-valid structured data back from the API. No parsing, no regex, no error handling for malformed responses. The `reasoning` field also serves as a form of chain-of-thought prompting, which tends to improve accuracy.

**The trade-offs** are clear:

- We need no training data or compute for model training.
- But we pay per request, we depend on an external service, and we have limited control over the model's behavior.
- Latency is much higher than a local model — seconds per article versus microseconds.
- For a production system with millions of articles, you would probably train a specialized model. For 500 articles or ad-hoc analysis, the API approach is dramatically faster to set up.